[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pandera-certified/notebooks/day-04-multi-backend-validation.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Validating Pandas, Polars, and Modin with the Same Schema
**certified-journeys / pandera-certified** · Multi-backend validation

> **Goal for today:** Define a single Pandera schema and use it to validate DataFrames across both the Pandas and Polars backends — observing how errors are reported in each.


In [ ]:
%pip install -q pandera polars


## Step 1 · How Pandera's multi-backend architecture works

Pandera 0.16+ ships separate sub-modules for each DataFrame backend:

| Module | Backend | Import |
|--------|---------|--------|
| `pandera` | Pandas (default) | `import pandera as pa` |
| `pandera.polars` | Polars | `import pandera.polars as pa_pl` |
| `pandera.modin` | Modin | `import pandera.modin as pa_md` |

Each module exposes the same `DataFrameSchema`, `Column`, and `Check` API — but the column dtypes map to the backend's native types (`pl.Int64`, `pl.Utf8`, etc. for Polars).

A **backend-agnostic validation function** accepts any supported DataFrame type and dispatches the correct schema at runtime.


In [ ]:
import pandas as pd
import pandera as pa

# Define a schema for a simple user events table using the Pandas backend
pandas_schema = pa.DataFrameSchema(
    {
        "user_id": pa.Column(int, pa.Check.ge(1), nullable=False),
        "event": pa.Column(str, pa.Check.isin(["click", "view", "purchase"])),
        "amount": pa.Column(float, pa.Check.ge(0.0), nullable=True),
    },
    name="UserEventsSchema",
    strict=False,  # allow extra columns
)

# Build a valid Pandas DataFrame
df_pandas = pd.DataFrame(
    {
        "user_id": [1, 2, 3],
        "event": ["click", "view", "purchase"],
        "amount": [0.0, None, 49.99],
    }
)

validated_pandas = pandas_schema.validate(df_pandas)
print("Pandas validation passed:")
print(validated_pandas)


**What just happened?**

- **`pa.DataFrameSchema`** declares columns by name, dtype, and one or more `Check` constraints.
- **`pa.Check.ge(1)`** means "greater than or equal to 1" — pandera ships dozens of built-in checks.
- **`nullable=True`** on `amount` allows `None` / `NaN` values without failing the check.
- The returned object is the original DataFrame, not a copy — pandera validates in-place and returns it.


## Step 2 · Validate the same logical table with a Polars schema

The Polars backend requires **Polars-native dtypes** (`pl.Int64`, `pl.Utf8`, `pl.Float64`) instead of Python built-ins. Other than the dtype tokens, the API is identical.

Key differences to watch:
- `pl.Utf8` is the Polars equivalent of `str` in Pandas schemas.
- Polars DataFrames are immutable — pandera returns a validated copy.
- `nullable` defaults to `False` in Pandera's Polars backend; set it explicitly.


In [ ]:
import polars as pl
import pandera.polars as pa_pl

# Mirror the same schema for the Polars backend
polars_schema = pa_pl.DataFrameSchema(
    {
        "user_id": pa_pl.Column(pl.Int64, pa_pl.Check.ge(1)),
        "event": pa_pl.Column(pl.Utf8, pa_pl.Check.isin(["click", "view", "purchase"])),
        "amount": pa_pl.Column(pl.Float64, pa_pl.Check.ge(0.0), nullable=True),
    },
    name="UserEventsSchema",
)

# Build the equivalent Polars DataFrame
df_polars = pl.DataFrame(
    {
        "user_id": [1, 2, 3],
        "event": ["click", "view", "purchase"],
        "amount": [0.0, None, 49.99],
    }
)

validated_polars = polars_schema.validate(df_polars)
print("Polars validation passed:")
print(validated_polars)


**What just happened?**

- **`pandera.polars`** exposes `DataFrameSchema`, `Column`, and `Check` — the same names as the Pandas module.
- Column dtypes are **Polars type objects** (`pl.Int64`, not `int`); mismatching them raises a `SchemaError`.
- Pandera validates each column's dtype first, then runs the `Check` predicates against Polars expressions.
- The return value is a validated Polars `DataFrame`.


## Step 3 · Write a backend-agnostic validation function

A real-world pipeline often needs to support both backends without duplicating schema definitions. The pattern is:

1. Detect the DataFrame type at runtime with `isinstance`.
2. Dispatch to the correct schema variant.
3. Return the validated DataFrame (same type as input).

This keeps validation logic centralized and makes switching backends a one-line change at the call site.


In [ ]:
from typing import Union
import pandas as pd
import polars as pl
import pandera as pa
import pandera.polars as pa_pl

# Pre-build both schema variants
_PANDAS_SCHEMA = pa.DataFrameSchema(
    {
        "user_id": pa.Column(int, pa.Check.ge(1)),
        "event": pa.Column(str, pa.Check.isin(["click", "view", "purchase"])),
        "amount": pa.Column(float, pa.Check.ge(0.0), nullable=True),
    }
)

_POLARS_SCHEMA = pa_pl.DataFrameSchema(
    {
        "user_id": pa_pl.Column(pl.Int64, pa_pl.Check.ge(1)),
        "event": pa_pl.Column(pl.Utf8, pa_pl.Check.isin(["click", "view", "purchase"])),
        "amount": pa_pl.Column(pl.Float64, pa_pl.Check.ge(0.0), nullable=True),
    }
)


def validate_user_events(
    df: Union[pd.DataFrame, pl.DataFrame]
) -> Union[pd.DataFrame, pl.DataFrame]:
    """Validate a user-events DataFrame regardless of backend."""
    if isinstance(df, pl.DataFrame):
        return _POLARS_SCHEMA.validate(df)
    elif isinstance(df, pd.DataFrame):
        return _PANDAS_SCHEMA.validate(df)
    else:
        raise TypeError(f"Unsupported DataFrame type: {type(df)}")


# Test with Pandas
pandas_result = validate_user_events(df_pandas)
print(f"Pandas backend  → {type(pandas_result).__name__}, shape {pandas_result.shape}")

# Test with Polars
polars_result = validate_user_events(df_polars)
print(f"Polars backend  → {type(polars_result).__name__}, shape {polars_result.shape}")


**What just happened?**

- **`isinstance` dispatch** routes each DataFrame to the pre-built schema for its backend.
- Both schemas encode the **same business rules** — only the dtype tokens differ.
- Pre-building schemas (module-level constants) avoids re-constructing them on every call, which matters at high throughput.
- Callers don't need to know which backend is active — they just call `validate_user_events`.


## Step 4 · Type annotations with `pandera.typing`

Pandera integrates with Python's type-annotation system via `pandera.typing`. You can annotate function arguments and return values with typed DataFrame classes that encode schema rules:

- `pandera.typing.pandas.DataFrame[MyModel]` — typed Pandas DataFrame
- `pandera.typing.polars.DataFrame[MyModel]` — typed Polars DataFrame

Combined with `@pa.check_types`, these annotations are enforced at runtime. This is especially useful in IDE-aware codebases where you want static type checking to catch schema mismatches early.


In [ ]:
import pandera as pa
from pandera.typing import pandas as papd
from pandera.typing import polars as papl
import pandera.polars as pa_pl
import pandas as pd
import polars as pl

# Define a DataFrameModel (class-based schema) for the Pandas backend
class UserEventsPandas(pa.DataFrameModel):
    user_id: pa.typing.Series[int] = pa.Field(ge=1)
    event: pa.typing.Series[str] = pa.Field(isin=["click", "view", "purchase"])
    amount: pa.typing.Series[float] = pa.Field(ge=0.0, nullable=True)

    class Config:
        strict = False


# Annotate a function using the typed DataFrame alias
@pa.check_types
def enrich_events_pandas(
    df: papd.DataFrame[UserEventsPandas],
) -> papd.DataFrame[UserEventsPandas]:
    """Add a 'processed' flag column (no-op enrichment for demo)."""
    return df.assign(processed=True)


result = enrich_events_pandas(df_pandas)
print("Enriched Pandas DataFrame:")
print(result)
print(f"\nReturn type: {type(result).__name__}")


**What just happened?**

- **`pa.DataFrameModel`** is the class-based alternative to `DataFrameSchema` — column specs become class attributes with `pa.Field` kwargs.
- **`papd.DataFrame[UserEventsPandas]`** is a generic type alias that carries the schema.
- **`@pa.check_types`** activates runtime enforcement of those type annotations — without it the annotations are purely decorative.
- The output passes the same schema check because `strict=False` allows the extra `processed` column.


## Step 5 · Comparing `SchemaError.failure_cases` across backends

When validation fails, Pandera raises a `SchemaError`. The `.failure_cases` attribute is a DataFrame containing each failing row's index and value. The **format** of `failure_cases` differs:

| Backend | `failure_cases` type | Index column |
|---------|---------------------|-------------|
| Pandas | `pd.DataFrame` | `index` (integer RangeIndex) |
| Polars | `pl.DataFrame` | `index` (null for Polars row positions) |

Understanding this difference is important for building generic error-reporting pipelines.


In [ ]:
import pandera as pa
import pandera.polars as pa_pl
import pandas as pd
import polars as pl

# --- Pandas failure cases ---
bad_pandas = pd.DataFrame(
    {
        "user_id": [1, -5, 3],       # row 1: user_id < 1 → will fail
        "event": ["click", "view", "hack"],  # row 2: 'hack' not in isin → will fail
        "amount": [0.0, None, 49.99],
    }
)

try:
    pandas_schema.validate(bad_pandas)
except pa.errors.SchemaError as exc:
    print("=== Pandas SchemaError ===")
    print(f"Column:        {exc.schema.name}")
    print(f"failure_cases type: {type(exc.failure_cases).__name__}")
    print(exc.failure_cases)
    print()

# --- Polars failure cases ---
bad_polars = pl.DataFrame(
    {
        "user_id": [1, -5, 3],
        "event": ["click", "view", "hack"],
        "amount": [0.0, None, 49.99],
    }
)

try:
    polars_schema.validate(bad_polars)
except Exception as exc:
    print("=== Polars SchemaError ===")
    print(f"Exception type: {type(exc).__name__}")
    # failure_cases is a Polars DataFrame in the Polars backend
    if hasattr(exc, 'failure_cases'):
        print(f"failure_cases type: {type(exc.failure_cases).__name__}")
        print(exc.failure_cases)
    else:
        print(str(exc)[:500])


**What just happened?**

- **Pandas backend** raises `pa.errors.SchemaError` with a `failure_cases` attribute that is a Pandas DataFrame.
- **Polars backend** raises its own `SchemaError` sub-class; `failure_cases` is a Polars `DataFrame` when present.
- Both `failure_cases` DataFrames include `index` and `failure_case` columns — but the Polars version may carry nulls for the index since Polars uses row positions rather than named index labels.
- **Production tip:** always catch by `pa.errors.SchemaError` (Pandas) or `pa_pl.errors.SchemaError` (Polars) separately in a backend-agnostic error handler.


## Step 6 · Lazy validation — collect all failures at once

By default, Pandera stops at the **first** failing column. Passing `lazy=True` runs **all** checks and collects every failure before raising a single `SchemaErrors` (note the plural) exception.

This is the recommended mode for data pipelines where you want a complete validation report.


In [ ]:
import pandera as pa
import pandas as pd

multi_error_schema = pa.DataFrameSchema(
    {
        "user_id": pa.Column(int, pa.Check.ge(1)),
        "score": pa.Column(float, pa.Check.between(0.0, 100.0)),
        "label": pa.Column(str, pa.Check.isin(["A", "B", "C"])),
    }
)

bad_df = pd.DataFrame(
    {
        "user_id": [-1, 2, 3],          # row 0 fails ge(1)
        "score": [50.0, 150.0, -5.0],   # rows 1 and 2 fail between(0, 100)
        "label": ["A", "Z", "C"],       # row 1 fails isin
    }
)

try:
    # lazy=True → collect ALL errors before raising
    multi_error_schema.validate(bad_df, lazy=True)
except pa.errors.SchemaErrors as errs:
    print("All failure cases collected in one pass:")
    print(errs.failure_cases)
    print(f"\nTotal failures: {len(errs.failure_cases)}")


**What just happened?**

- `lazy=True` switches from **fail-fast** to **collect-all** mode.
- The exception type changes to **`pa.errors.SchemaErrors`** (plural) — its `.failure_cases` is a DataFrame with one row per violation.
- Every failure across **all three columns** is surfaced in a single exception — 4 violations in this example.
- The `.data` attribute on `SchemaErrors` gives you the original (unmodified) input DataFrame for inspection.


In [ ]:
# Challenge: Backend-agnostic validator with lazy mode
#
# 1. Create a DataFrameModel called `OrderSchema` with columns:
#    - order_id: int, must be >= 1
#    - quantity: int, must be >= 1
#    - price: float, must be > 0.0
#    - status: str, must be one of ["pending", "shipped", "delivered"]
#
# 2. Write a function `validate_orders(df)` that:
#    - Accepts a Pandas or Polars DataFrame
#    - Validates using lazy=True
#    - On SchemaErrors, prints a summary (column + failure_case + index)
#       and returns None
#    - On success, returns the validated DataFrame
#
# 3. Test it with a bad Pandas DataFrame that has at least 2 violated columns.

# Your solution here


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `pandera.polars` | Separate sub-module; same API, Polars-native dtypes |
| Dtype tokens | `pl.Int64`, `pl.Utf8`, `pl.Float64` instead of `int`, `str`, `float` |
| Backend-agnostic function | `isinstance` dispatch to the correct schema variant |
| `pa.DataFrameModel` | Class-based schema; pairs with `@pa.check_types` for typed annotations |
| `papd.DataFrame[Model]` vs `papl.DataFrame[Model]` | Generic aliases for Pandas vs Polars typed inputs/outputs |
| `failure_cases` | Backend-native DataFrame (Pandas or Polars) with `index` + `failure_case` cols |
| `lazy=True` | Collect ALL violations before raising `SchemaErrors` (plural) |

> **Tip:** Keep your schema definitions as module-level constants (not rebuilt per call) to avoid paying construction cost at high throughput.

---
## What's next
**Day 5** → Apply `@pa.check_input` and `@pa.check_output` decorators to validate DataFrames entering and leaving pipeline functions automatically.

Mark Day 4 complete in your [tracker](../index.html).
